# Phase 2 — Notebook 03: Embedding

**Goal:** Load `BAAI/bge-large-en-v1.5`, verify 1024-dim output, validate L2 normalization, benchmark batch speed, and test the query vs. document prefix difference.

**Packages used:** `sentence-transformers`, `numpy`, `pandas`

## 1. Setup

In [ ]:
import sys, time
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from core.config import get_settings
settings = get_settings()

print(f"Provider : {settings.embedding_provider}")
print(f"Model    : {settings.embedding_model}")
print(f"Dim      : {settings.embedding_dim}")
print(f"Batch    : {settings.embedding_batch_size}")

## 2. Load Model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(f"Loading {settings.embedding_model} ...")
t0 = time.time()
model = SentenceTransformer(settings.embedding_model)
print(f"Loaded in {time.time() - t0:.2f}s")
print(f"Max sequence length: {model.max_seq_length}")

## 3. Verify Output Dimension

In [ ]:
sample_text = "Retrieval-augmented generation combines document retrieval with LLM generation."
vec = model.encode(sample_text, normalize_embeddings=True)

print(f"Output dim      : {len(vec)}")
print(f"Expected dim    : {settings.embedding_dim}")
assert len(vec) == settings.embedding_dim, \
    f"Dim mismatch: got {len(vec)}, expected {settings.embedding_dim}"
print("✅ Dimension matches EMBEDDING_DIM in .env")

## 4. Validate L2 Normalization

In [ ]:
# L2 norm of a normalised vector must equal 1.0
norm = np.linalg.norm(vec)
print(f"L2 norm (normalized)  : {norm:.8f}")
assert abs(norm - 1.0) < 1e-6, f"Vector is not L2-normalised (norm={norm})"
print("✅ L2 normalisation confirmed")

# Without normalization — norm will differ from 1.0
vec_raw = model.encode(sample_text, normalize_embeddings=False)
norm_raw = np.linalg.norm(vec_raw)
print(f"L2 norm (raw, no norm): {norm_raw:.4f}")

## 5. Query vs. Document Embedding Difference

BGE uses `"Represent this sentence for searching relevant passages: "` as a query prefix. Without it, query embeddings may not align well with document embeddings.

In [ ]:
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

query = "What is retrieval-augmented generation?"
doc   = "Retrieval-augmented generation (RAG) is a technique that combines a retrieval system with a generative language model."

# With query instruction (correct approach)
vec_query_with    = model.encode(QUERY_INSTRUCTION + query, normalize_embeddings=True)
vec_query_without = model.encode(query, normalize_embeddings=True)
vec_doc           = model.encode(doc,   normalize_embeddings=True)

sim_with    = float(np.dot(vec_query_with,    vec_doc))
sim_without = float(np.dot(vec_query_without, vec_doc))

print(f"Similarity WITH query instruction   : {sim_with:.4f}")
print(f"Similarity WITHOUT query instruction: {sim_without:.4f}")
print()
if sim_with > sim_without:
    print("✅ Query instruction improves alignment")
else:
    print("ℹ️  No improvement — model may not require instruction prefix")

## 6. Batch Embedding Benchmark

In [ ]:
import pandas as pd

# Generate synthetic corpus
corpus = [
    f"This is document number {i} about topic {i % 10} with some additional text for realistic length."
    for i in range(200)
]

batch_sizes = [8, 16, 32, 64, 128]
results = []

for batch_size in batch_sizes:
    t0 = time.time()
    embeddings = model.encode(
        corpus,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    elapsed = time.time() - t0
    results.append({
        "batch_size": batch_size,
        "total_docs": len(corpus),
        "time_s": round(elapsed, 3),
        "docs_per_sec": round(len(corpus) / elapsed, 1),
    })

df = pd.DataFrame(results)
print("Batch embedding benchmark (200 docs):")
print(df.to_string(index=False))
print(f"\nSelected batch size: {settings.embedding_batch_size}")

## 7. Cosine Similarity Sanity Check

In [ ]:
pairs = [
    ("The cat sat on the mat.",   "A feline rested on a rug."),           # similar
    ("Deep learning for NLP",     "Convolutional neural networks for CV"),# related
    ("Recipe for chocolate cake", "The history of the Roman Empire"),     # unrelated
]

print("Cosine similarity pairs:")
for a, b in pairs:
    va = model.encode(a, normalize_embeddings=True)
    vb = model.encode(b, normalize_embeddings=True)
    sim = float(np.dot(va, vb))
    print(f"  {sim:.4f}  |  '{a[:40]}' ↔ '{b[:40]}'")

## 8. Embedding Model Version Check

In [ ]:
from core.constants import MODEL_DIM_MAP

model_name    = settings.embedding_model
model_version = settings.embedding_model_version
expected_dim  = MODEL_DIM_MAP.get(model_name)

print(f"Model name      : {model_name}")
print(f"Model version   : {model_version}")
print(f"Expected dim    : {expected_dim}")
print(f"Config dim      : {settings.embedding_dim}")
print(f"Actual dim      : {len(vec)}")

assert expected_dim == settings.embedding_dim == len(vec), "Dimension mismatch across config, constants, and model"
print("\n✅ Model, config, and output dim are all consistent")

## Summary

| Check | Result |
|---|---|
| Output dim = 1024 | ✅ |
| L2 normalised | ✅ |
| Query instruction improves alignment | ✅ |
| Optimal batch size | see benchmark above |
| Model/config/dim consistent | ✅ |

**Next:** `04_embedding_cache.ipynb` — Redis caching of embeddings.